# Clase 11: Manejo de archivos FITS

FITS (Flexible Image Transport System) es el formato de archivo estándar más utilizado en astronomía para el almacenamiento, la transmisión y la manipulación de datos científicos. Diseñado específicamente para ser independiente de la plataforma y duradero a largo plazo, un archivo FITS consta de una o más secciones que incluyen un encabezado (header) en texto legible con metadatos en formato `key`-`value`, seguido de datos binarios que pueden representar imágenes multidimensionales, espectros o tablas de datos. Este formato permite incluir información sobre las condiciones de observación, las coordenadas celestiales y la calibración.

## Visualizando imágenes en archivos FITS

Imagen procesada del disco de escombros de Fomalhaut basada en imágenes tomadas por el Hubble Space Telescope. https://github.com/saint-germain/cazandoplanetas

In [ ]:
from astropy.io import fits

In [ ]:
fomalhaut=fits.open("https://github.com/saint-germain/cazandoplanetas/raw/66fb7ce82feb5e983e465b7cfd1a638d8bc3f038/Intermedio/PCA_FOMAL_full_centroids.fits")

In [ ]:
fomalhaut.info()

In [ ]:
hst_img=fomalhaut[0].data

In [ ]:
hst_img

In [ ]:
hst_img.max()

In [ ]:
plt.imshow(hst_img)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(hst_img*(hst_img>-0.0045)*(hst_img<0.004),cmap="gist_heat")
plt.scatter(1173,1205,s=1000,facecolors='none', edgecolors='b',lw=2)
plt.xlim(900,1250)
plt.ylim(1100,1400)

### Explorando un cubo de datos en un archivo FITS

Datos de espectros de CO galáctico (CO J = 1–0 line, vrest = 115.27 GHz) tomados por el radiotelescopio MINI (Crédito: Leonardo Bronfman, U. de Chile).

In [ ]:
!wget https://github.com/saint-germain/astropy/raw/refs/heads/master/southgal_fixbadc.fits

In [ ]:
# esta línea da un warning debido a que es un fits no estándar
cubo = fits.open("southgal_fixbadc.fits") #abrir objeto cubo de datos

In [ ]:
cubo.info()

In [ ]:
cubo[0].data

In [ ]:
cubo[0].header

In [ ]:
def values(h,j):
    # construye los ejes en velocidad, longitud y latitud con la información en el header
	N=h['NAXIS'+str(j)];
	val=np.zeros(N);
	for i in range(0,N):
		val[i] = (i+1-float(h['CRPIX'+str(j)]))*float(h['CDELT'+str(j)]) + float(h['CRVAL'+str(j)]);
	return val;

data 	= cubo[0].data #extraer matriz de datos
header	= cubo[0].header #extraer el header del archivo fits
#print header

#Estos seran los tres arreglos con los valores reales de los tres ejes del cubo
velocidad=values(header,1)
longitud=values(header,2)
latitud=values(header,3)

In [ ]:
# Escogemos algún valor arbitrario para longitud, latitud
# para extraer el espectro para ese apuntamiento
i_l=-1
i_b=-1
T = data[i_b][i_l][:]

In [ ]:
lon=longitud[i_l]
lat=latitud[i_b]
plt.plot(velocidad,T)
plt.title(r'Espectro de la emisión de CO $J=1-0$ para ($l,b$)=(%i,%i)'%(lon,lat))

In [ ]:
maxeix=np.argmax(data,axis=2)
varr=np.array([velocidad[i] for i in maxeix.ravel()]).reshape(maxeix.shape)
plt.figure(figsize=(20,5))
plt.imshow(varr,cmap='bwr',extent=[longitud[0],longitud[-1],latitud[0],latitud[-1]])

También existe una librería llamada spectral_cube que puede procesar automáticamente este tipo de cubos de datos en formato fits (estandarizados): https://learn.astropy.org/tutorials/FITS-cubes.html

## Tablas (Votable) y WCS

Exploración de datos xml en formato Votable: el caso de las estrellas en el grupo móvil Lower Centaurus Crux (LCC).

Ir a SIMBAD, buscar "lcc", children objects, descargar Votable

Enlace:
http://simbad.u-strasbg.fr/simbad/sim-id?Ident=NAME%20LCC&NbIdent=query_hlinks&Coord=12%2019-57.1&parents=1&children=1938&submit=children&siblings=19&hlinksdisplay=h_all

In [ ]:
!wget https://github.com/saint-germain/astropy/raw/refs/heads/master/lcc_simbad_votable.xml

In [ ]:
from astropy.io.votable import parse_single_table
votable = parse_single_table("lcc_simbad_votable.xml").to_table()

In [ ]:
votable

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.subplot()
plt.scatter(votable['RA_d'],votable['DEC_d'])
plt.xlabel(r'RA')
plt.ylabel(r'Dec')

Las coordenadas mundiales (World Coordinates) sirven para ubicar una medida en un espacio de parámetros multidimensional. Un Sistema de Coordenadas Mundiales (WCS en inglés) especifica las coordenadas físicas, o mundiales (WC en inglés), que se adjuntan a cada píxel o spaxel de una imagen o matriz de $N$ dimensiones. Se ha desarrollado un elaborado conjunto de normas y convenciones para el formato FITS (Flexible Image Transport System) (Wells et al. 1981). Un ejemplo típico de WCS es la especificación de la Ascensión Recta (AR) y la Declinación (Dec) en el cielo asociada a una determinada ubicación de píxel o vóxel en una imagen celeste bidimensional (Greisen y Calabretta 2002; Calabretta y Greisen 2002).

Hay dos formas principales de inicializar un objeto WCS: con un diccionario de Python (o un objeto tipo diccionario, como la cabecera de un archivo FITS) o con listas de Python. En este ejemplo, inicializaremos un objeto astropy.wcs.WCS con dos dimensiones, como sería necesario para representar una imagen.

Adaptado y traducido de https://learn.astropy.org/tutorials/celestial_coords1.html

In [ ]:
from astropy.wcs import WCS
# world coordinate system https://learn.astropy.org/tutorials/celestial_coords1.html

In [ ]:
wcs_input_dict = {
    'CTYPE1': 'RA',
    'CUNIT1': 'deg',
    'CTYPE2': 'DEC',
    'CUNIT2': 'deg'
}
wcs_helix = WCS(wcs_input_dict)

In [ ]:
wcs_helix

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.subplot(projection=wcs_helix)
plt.scatter(votable['RA_d'],votable['DEC_d'])
plt.xlabel(r'RA')
plt.ylabel(r'Dec')
overlay = ax.get_coords_overlay('icrs')
overlay.grid(color='black', ls='dotted')

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.subplot(projection=wcs_helix)
plt.scatter(votable['RA_d'],votable['DEC_d'])
plt.xlabel(r'RA')
plt.ylabel(r'Dec')
overlay = ax.get_coords_overlay('galactic')
overlay.grid(color='black', ls='dotted')

## Encontrar y Descargar una Imagen de Herschel

Vamos a ver cómo aparece la Nube Menor de Magallanes con observaciones de emisión a 350 micras de Herschel para rastrear la emisión térmica (en continuo del polvo). Esto se puede hacer con [astroquery](http://www.astropy.org/astroquery/).

Podemos consultar los datos por misión, dar un vistazo rápido a la tabla de resultados y descargar los datos después de seleccionar una longitud de onda o filtro específico.

Como estamos buscando datos de Herschel provenientes de una misión de la ESA, utilizaremos la clase [astroquery.ESASky](http://astroquery.readthedocs.io/en/latest/esasky/esasky.html).

Específicamente, el método `ESASKY.query_region_maps()` nos permite buscar una región específica del cielo utilizando ya sea un objeto SkyCoord de Astropy o una cadena de texto que especifique el nombre de un objeto. En este caso, podemos simplemente buscar la SMC. También se puede especificar un radio de búsqueda alrededor del objeto.

[Tutorial original](https://learn.astropy.org/tutorials/FITS-cubes.html)

In [ ]:
!pip install astroquery

In [ ]:
from astroquery.esasky import ESASky
from astropy import units as u
import numpy as np
from astroquery.utils import TableList
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
maps_list = ESASky.list_maps()
print(maps_list)

In [ ]:
# Query for Herschel data in a 1 degree radius around the SMC
result = ESASky.query_region_maps("SMC", radius=1 * u.deg, missions="Herschel")

print(result)

In [ ]:
# Query for Herschel data in a 1 degree radius around the SMC
result = ESASky.query_region_maps("SMC", radius=1 * u.deg, missions="Herschel")

print(result)

Aquí, el resultado es un `TableList` que contiene 24 productos de datos de Herschel que pueden descargarse. Podemos ver qué información está disponible en este `TableList` examinando las `keys` de la tabla de Herschel.

In [ ]:
result["HERSCHEL"].keys()

Queremos encontrar una imagen de 350 micras, así que necesitamos examinar con más detalle los filtros utilizados para estas observaciones.

In [ ]:
result["HERSCHEL"]["filter"]

Afortunadamente para nosotros, hay una observación realizada con tres filtros: 250, 350 y 500 micras. Este es el objeto que queremos descargar. Una forma de hacerlo es creando una máscara booleana para seleccionar la entrada de la tabla correspondiente al filtro deseado. Luego, el método `ESASky.get_maps()` descargará nuestros datos proporcionando un argumento de tipo `TableList`.

La siguiente descarga puede tomar varios minutos.

In [ ]:
filters = result["HERSCHEL"]["filter"].astype(
    str
)  # Convert the list of filters from the query to a string

# Construct a boolean mask, searching for only the desired filters
mask = np.array(["250, 350, 500" == s for s in filters], dtype="bool")

# Re-construct a new TableList object containing only our desired query entry
target_obs = TableList(
    {"HERSCHEL": result["HERSCHEL"][mask]}
)  # This will be passed into ESASky.get_maps()

IR_images = ESASky.get_maps(target_obs)  # Download the images
IR_images["HERSCHEL"][0][
    "350"
].info()  # Display some information about the 350 micron image

Vamos a extraer únicamente la información de WCS y datos de la imagen de 350 micrones.

In [ ]:
herschel_header = IR_images["HERSCHEL"][0]["350"]["image"].header
herschel_wcs = WCS(IR_images["HERSCHEL"][0]["350"]["image"])  # Extract WCS information
herschel_imagehdu = IR_images["HERSCHEL"][0]["350"]["image"]  # Extract Image data
print(herschel_wcs)

Con esto, podemos mostrar esta imagen utilizando matplotlib con [WCSAxes](http://docs.astropy.org/en/stable/visualization/wcsaxes/index.html) y el objeto `LogNorm()` para aplicar una escala logarítmica a nuestra imagen.

In [ ]:
from matplotlib.colors import LogNorm


# Set Nans to zero
himage_nan_locs = np.isnan(herschel_imagehdu.data)
herschel_data_nonans = herschel_imagehdu.data
herschel_data_nonans[himage_nan_locs] = 0

# Initiate a figure and axis object with WCS projection information
fig = plt.figure(figsize=(18, 12))
ax = fig.add_subplot(111, projection=herschel_wcs)

# Display the moment map image
im = ax.imshow(herschel_data_nonans, cmap="viridis", norm=LogNorm(vmin=2, vmax=50))
# ax.invert_yaxis() # Flips the Y axis

# Add axes labels
ax.set_xlabel("Right Ascension", fontsize=16)
ax.set_ylabel("Declination", fontsize=16)
ax.grid(color="white", ls="dotted", lw=2)

# Add a colorbar
cbar = plt.colorbar(im, pad=0.07)
cbar.set_label(
    "".join(["Herschel 350" r"$\mu$m ", "(", herschel_header["BUNIT"], ")"]), size=16
)

# Overlay set of Galactic Coordinate Axes
overlay = ax.get_coords_overlay("galactic")
overlay.grid(color="black", ls="dotted", lw=1)
overlay[0].set_axislabel("Galactic Longitude", fontsize=14)
overlay[1].set_axislabel("Galactic Latitude", fontsize=14)

Ejercicio: Con los contenidos del catálogo de nubes oscuras  de Barnard [VII/220A/barnard](https://vizier.cds.unistra.fr/viz-bin/VizieR-3?-source=VII/220A/barnard) haga un mosaico con los objetos del catálogo que tengan imágenes de Herschel disponibles.